## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web or running the code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions(often a JSON schema)

2. A function or coroutine to execute

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

model= init_chat_model("qwen/qwen3.8-27b",model_provider="groq")
response= model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don\'t actually "talk" in the human sense of using language to convey complex thoughts or abstract ideas. Instead, they **mimic sounds** they hear, primarily through a process called **vocal mimicry**. Here’s why and how it happens:\n\n### 1. **Vocal Mimicry**\nParrots have a specialized vocal organ called the **syrinx**, located at the base of their trachea (windpipe). This organ allows them to produce a wide range of sounds, including:\n- Bird calls\n- Whistles\n- Human speech\n- Mechanical sounds (like doorbells or alarms)\n\nThey can replicate these sounds with surprising accuracy because of the flexibility of their syrinx and their strong control over breathing and mouth movements.\n\n### 2. **Social Bonding and Communication**\nIn the wild, parrots are highly social birds that live in flocks. They use a variety of calls to:\n- Stay in contact with the flock\n- Warn of predators\n- Establish territory\n- Reinforce social bonds\n\nWhen kept as pets, parro

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at a location """
    return f"It's suuny in {location}"

model_with_tools= model.bind_tools([get_weather])

In [4]:
response= model_with_tools.invoke("Whats the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    #View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'qtjs9vrd3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.069246846, 'completion_tokens_details': None, 'prompt_time': 0.022822169, 'prompt_tokens_details': None, 'queue_time': 0.04912536, 'total_time': 0.092069015}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a10047-3251-7132-9c6d-d66126923002-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'qtjs9vrd3', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}
Tool: get_weather
Args:{'location': 'Boston'}


## Tool Execution Loops

In [6]:
#Step 1: Model generated tool calls

messages= [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg= model_with_tools.invoke(messages)
messages.append(ai_msg)


#Step 2: Execute tools and collect results

for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments

    tool_result= get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response= model_with_tools.invoke(messages)
print(final_response.text)

# "The current weather in Boston is 72F and sunny."

It's sunny in Boston! ☀️


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kasfjb6ep', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 277, 'total_tokens': 303, 'completion_time': 0.069138995, 'completion_tokens_details': None, 'prompt_time': 0.022021339, 'prompt_tokens_details': None, 'queue_time': 0.05393212, 'total_time': 0.091160334}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1004c-0a84-7d22-9823-ca479ec9d7bd-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'kasfjb6ep', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 26, 'total_tokens': 303}),
 ToolMessage(content="It's suuny i